# Add new tables to test dataset

In [1]:
import ngio
import pandas as pd

/Users/joel/Documents/Code/napari-ome-zarr-navigator/.pixi/envs/local-dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
zarr_plate = "/Users/joel/Documents/TestDatasets/navigator_test_data/11262587/20200812-CardiomyocyteDifferentiation14-Cycle1.zarr"
zarr_plate = "/Users/joel/Documents/TestDatasets/navigator_test_data/11262587/20200812-CardiomyocyteDifferentiation14-Cycle1_mip.zarr"

In [3]:
plate = ngio.open_ome_zarr_plate(zarr_plate)
plate

Plate([rows x columns] (1 x 2))

In [15]:
condition_table_df = pd.DataFrame(
    {
        "well": ["B03", "B05"],
        "row": ["B", "B"],
        "column": ["03", "05"],
        "differentiation_timepoint": ["day 0", "day 6"],
    }
)

In [16]:
condition_table = ngio.tables.ConditionTable(condition_table_df)
condition_table

ConditionTableV1

In [17]:
plate.add_table(
    name="differentiation_timepoint",
    table=condition_table,
    backend="csv",
    overwrite=True,
)

In [29]:
roi_tables = ["FOV_ROI_table", "well_ROI_table"]
for roi_table in roi_tables:
    for well in plate.wells_paths():
        image = "0"
        ome_zarr_container = ngio.open_ome_zarr_container(f"{zarr_plate}/{well}/{image}")
        roi_table_obj = ome_zarr_container.get_table(roi_table)
        ome_zarr_container.add_table(
            name=f"{roi_table}_csv",
            table=roi_table_obj,
            backend="csv",
            overwrite=True,
        )

In [30]:
for well in plate.wells_paths():
    image = "0"
    ome_zarr_container = ngio.open_ome_zarr_container(f"{zarr_plate}/{well}/{image}")
    masking_roi_table =ome_zarr_container.build_masking_roi_table(label="nuclei")
    ome_zarr_container.add_table(
        name="masking_roi_table_nuclei",
        table=masking_roi_table,
        backend="csv",
        overwrite=True,
    )



In [35]:
feature_table_name = "measurements"
for well in plate.wells_paths():
    image = "0"
    ome_zarr_container = ngio.open_ome_zarr_container(f"{zarr_plate}/{well}/{image}")
    feature_table_df = ome_zarr_container.get_table(feature_table_name).dataframe
    feature_table = ngio.tables.FeatureTable(table_data=feature_table_df, reference_label="nuclei")
    ome_zarr_container.add_table(
        name="measurements_csv",
        table=feature_table,
        backend="csv",
        overwrite=True,
    )


/Users/joel/Documents/Code/napari-ome-zarr-navigator/.pixi/envs/local-dev/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/Users/joel/Documents/Code/napari-ome-zarr-navigator/.pixi/envs/local-dev/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [7]:
plate.concatenate_image_tables(name="condition").dataframe

,row,col,drug,concentration,extra1,extra2,column,path_in_well
0,B,03,drug1,0.1,True,example,03,0
1,B,03,drug2,10.0,False,example,03,0
0,B,05,drug1,5.0,True,example,05,0
